# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryam884/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*


This workflow is intended to help content teams prioritize which pages should be reviewed first. The recommendations are based on observed historical content signals and should be used as decision-support rather than automatic decisions. The results may not generalize to other datasets without additional validation because the model was evaluated on a limited dataset.

In [10]:
%cd flyrank-ml-internship

[Errno 2] No such file or directory: 'flyrank-ml-internship'
/content/flyrank-ml-internship


In [11]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Recreate the target from Week 5
df["target"] = (
    (df["days_since_last_update"] > 365) &
    (df["trend_direction"] == "down")
).astype(int)

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,target
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,0
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,0
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,0
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,0


In [12]:
import pandas as pd

# Create ranked action queue
queue = df.copy()

queue["Reason Code"] = "Review for content refresh"

queue["Recommended Action"] = queue["target"].map({
    1: "Refresh content",
    0: "Monitor"
})

queue = queue.sort_values(
    by="days_since_last_update",
    ascending=False
)

queue = queue[[
    "days_since_last_update",
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "Recommended Action",
    "Reason Code"
]]

print(queue.head())

       days_since_last_update  impressions_90d  sessions_90d    ctr  \
26242                     373               35             1    0.0   
4606                      373                1             1  100.0   
29384                     373                2             1    0.0   
24216                     372                2             2    0.0   
18440                     372                1             2    0.0   

       avg_position Recommended Action                 Reason Code  
26242           7.5    Refresh content  Review for content refresh  
4606            1.0            Monitor  Review for content refresh  
29384          32.5    Refresh content  Review for content refresh  
24216           7.0    Refresh content  Review for content refresh  
18440          35.0            Monitor  Review for content refresh  


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This workflow is intended to help content teams prioritize which pages should be reviewed first. The recommendations are based on observed historical content signals and should be used as decision-support rather than automatic decisions. The results may not generalize to other datasets without additional validation because the model was evaluated on a limited dataset.

In [13]:
intended_use = {
    "Users": "Content reviewers",
    "Purpose": "Prioritize pages for review",
    "Automation": "Decision-support only"
}

print(intended_use)


{'Users': 'Content reviewers', 'Purpose': 'Prioritize pages for review', 'Automation': 'Decision-support only'}


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on any recommendation, a reviewer should verify the current content quality, business priorities, search intent, and recent performance trends. Recommendations should not automatically publish, remove, or substantially rewrite content without human approval. Final editorial decisions should always remain under human control.

In [14]:
human_review = [
    "Check content quality",
    "Review search intent",
    "Verify recent performance",
    "Confirm business priority"
]

no_go = [
    "Do not automatically publish content.",
    "Do not automatically delete pages.",
    "Do not automatically rewrite content."
]

print("Human review checklist:")
for item in human_review:
    print("-", item)

print("\nNo-go list:")
for item in no_go:
    print("-", item)


Human review checklist:
- Check content quality
- Review search intent
- Verify recent performance
- Confirm business priority

No-go list:
- Do not automatically publish content.
- Do not automatically delete pages.
- Do not automatically rewrite content.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The model should be monitored over time. Retraining may be appropriate if content patterns change, accuracy decreases, new features become available, or the recommendation quality consistently declines during human review.

In [15]:
triggers = [
    "Model accuracy decreases",
    "Content patterns change",
    "New historical data becomes available",
    "Recommendations require frequent manual correction"
]

for trigger in triggers:
    print("-", trigger)


- Model accuracy decreases
- Content patterns change
- New historical data becomes available
- Recommendations require frequent manual correction


## 5. Exports for the paper

The ranked recommendation queue is exported for use in the research paper. This exported file allows later notebooks to reproduce the reported recommendations without manually recreating the workflow.


In [16]:
from pathlib import Path

# Create output directory
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Export ranked queue
queue.to_csv(output_dir / "ranked_action_queue.csv", index=False)

print("Exported:")
print(output_dir / "ranked_action_queue.csv")


Exported:
work/outputs/ranked_action_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.